# Fine-tune Qwen2.5-VL-7B — CMS-1500 **SERVICE** model (box 24 line items)

v2 region-specialized model **3 of 3**. One of three *independent* single-band models, each
fine-tuned and deployed separately and called **concurrently** as three Modal functions
(latency = the slowest model, not the sum):

- **top** — boxes 1–23 (`finetune_qwen2_5_vl_7b_hcfa_top.ipynb`).
- **bottom** — boxes 25–33 (`finetune_qwen2_5_vl_7b_hcfa_bottom.ipynb`).
- **service** ← this notebook — box 24 line-item table, one set of keys per line (`box_24_service_lines[i].*`: dates, place of service, procedure code, modifiers, diagnosis pointers, charges, units, rendering NPI).

**Base model: `Qwen/Qwen2.5-VL-7B-Instruct`** (the step up from v1's 3B).

**Why a dedicated model?** The table is the one *variable-length, tabular* region — a
different shape from the single-value boxes — fed at full 300-DPI detail so procedure codes,
modifiers, line NPIs and charges (v1's weak dense small-text fields) stay legible, and its
JSON almost always parses. Independent weights mean a trivially parallel Modal call.

Same QLoRA recipe as v1; the structural changes are **7B base + single-band crop**. The
band is selected by the `MODEL` constant in cell 4. Run top to bottom on Colab (L4 24 GB or
better — 7B is ~2× the VRAM of 3B).

## 1. Install dependencies

In [ ]:
# Qwen2.5-VL needs transformers>=4.49. trl/peft/bitsandbytes for QLoRA SFT.
%pip install -q -U "transformers>=4.49" "trl>=0.12" "peft>=0.13" accelerate bitsandbytes datasets qwen-vl-utils
# hcfa_eval (+ hcfa_synth schema) so we can score with the repo's harness in-notebook:
%pip install -q "git+https://github.com/chrscato/hcfa-1500-reader.git"

## 2. Detect GPU → pick a hardware profile

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # reclaim fragmented VRAM
import torch

assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> GPU."
name = torch.cuda.get_device_name(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
BF16 = torch.cuda.is_bf16_supported()          # False on T4 (Turing); True on L4 / A100
DTYPE = torch.bfloat16 if BF16 else torch.float16

# Profiles are tuned DOWN from the 3B notebook: 7B roughly doubles VRAM, so the pixel caps
# are lower. That's fine — a single band crop is small, so even these caps give high
# effective DPI on the region. Resolution is still the main accuracy lever; raise the cap
# if you have headroom.
if total_gb < 20:          # T4 ~15 GB (free) — 7B is TIGHT here; low res, 1 epoch. Prefer L4+.
    PROFILE = dict(tier="T4", max_pixels_mult=448, epochs=1, batch=1, grad_accum=8, eval="no")
elif total_gb < 34:        # L4 ~24 GB (Colab Pro) — comfortable for 7B QLoRA  <-- recommended
    PROFILE = dict(tier="L4", max_pixels_mult=1024, epochs=3, batch=1, grad_accum=8, eval="epoch")
else:                      # A100 40 GB+ — full res
    PROFILE = dict(tier="A100", max_pixels_mult=1280, epochs=3, batch=1, grad_accum=8, eval="epoch")

print(f"GPU: {name} ({total_gb:.0f} GB) | bf16={BF16} -> {DTYPE}")
print("profile:", PROFILE)

## 3. Authenticate + load the dataset

`login()` opens a token box — paste a token from https://huggingface.co/settings/tokens
(a **read** token is enough to load; you need **write** later to push the adapter).

In [ ]:
from huggingface_hub import login
login()  # paste your HF token

from datasets import load_dataset

REPO_ID = "catochris/hcfa-1500-v2"   # v2 dataset (region crops happen in-notebook)
ds = load_dataset(REPO_ID)
print(ds)

In [ ]:
# Peek at one example: an image, the prompt, and the flat-JSON target.
ex = ds["train"][0]
print("columns:", ds["train"].column_names)
print("tier:", ex["tier"], "| sample_id:", ex["sample_id"])
print("image size:", ex["image"].size)
print("target (head):", ex["target"][:300], "...")
ex["image"]

## 4. Region configuration (`MODEL` selects the band)

The single cell that makes this a *region* model. We:

1. set `MODEL` (`"top"` here; the other two notebooks use `"bottom"` / `"service"`),
2. crop each stored full-page image down to this model's band,
3. subset the flat-JSON `target` to just the keys in that band,
4. build one training row per form (single band → no expansion, no region tag).

Band pixel rectangles are hardcoded for the fixed CMS-1500 layout (produced by
`hcfa_synth.regions.region_bands_px(300)`), so the notebook needs no template PDF. The
pure-Python helpers `region_of_key` / `MODEL_BANDS` come from the installed `hcfa_synth`.

In [ ]:
# ---- v2 region configuration -------------------------------------------------
import json
from datasets import Dataset
from hcfa_synth import regions  # pure-Python helpers; no template PDF needed here

MODEL = "service"                      # "top" | "bottom" | "service" — this is the SERVICE notebook
BAND = regions.MODEL_BANDS[MODEL][0]   # each v2 model owns exactly one band
print("MODEL:", MODEL, "| band:", BAND)

# Prompt for this band. Each single-band model only ever sees one prompt; we keep all three
# here so the cell is identical across the three notebooks (only MODEL changes).
BAND_PROMPT = {
    "top": (
        "Extract the TOP section of this CMS-1500 (HCFA) claim form — boxes 1-23: "
        "insurance carrier, patient and insured details, condition, dates, diagnoses, and "
        "referring provider — as a single flat JSON object. "
        'Use "" for any blank field. Return only the JSON.'
    ),
    "bottom": (
        "Extract the BOTTOM section of this CMS-1500 (HCFA) claim form — boxes 25-33: "
        "federal tax ID, patient account number, accept-assignment, charges, physician "
        "signature, service facility, and billing provider — as a single flat JSON object. "
        'Use "" for any blank field. Return only the JSON.'
    ),
    "service": (
        "Extract the service-line table (box 24) of this CMS-1500 (HCFA) claim form as a "
        "single flat JSON object. Emit one set of keys per line using the paths "
        "box_24_service_lines[i].<field>. "
        'Use "" for any blank field. Return only the JSON.'
    ),
}
PROMPT = BAND_PROMPT[BAND]

# Pixel bands at 300 DPI for the fixed CMS-1500 layout (page 2550x3300). Static; generated
# by hcfa_synth.regions.region_bands_px(300). Hardcoded so this runs in Colab without the
# template PDF. Bands overlap slightly so glyphs on a cut aren't sliced.
_BANDS_300 = {
    "top":     (0, 0, 2550, 2254),
    "service": (0, 2154, 2550, 2847),
    "bottom":  (0, 2747, 2550, 3300),
}
_PAGE_W, _PAGE_H = 2550, 3300


def crop_for_model(image):
    """Crop this model's band out of a full-page form, scaling if the image was resized."""
    x0, y0, x1, y1 = _BANDS_300[BAND]
    sx, sy = image.width / _PAGE_W, image.height / _PAGE_H
    return image.crop((int(x0 * sx), int(y0 * sy), int(x1 * sx), int(y1 * sy)))


def target_for_model(full_target_json):
    """Flat-JSON target subset whose keys fall in this model's band."""
    flat = json.loads(full_target_json)
    return json.dumps({k: v for k, v in flat.items() if regions.region_of_key(k) == BAND})


# One lightweight row per form (single band). Images are cropped lazily in the collator.
def build_index(split):
    return Dataset.from_list([{"idx": i, "split": split} for i in range(len(ds[split]))])


train_index = build_index("train")
val_index = build_index("validation")
print("train rows:", len(train_index), "| val rows:", len(val_index))

# Peek: confirm the crop + target subset look right.
_ex = ds["train"][0]
print("crop size:", crop_for_model(_ex["image"]).size)
print("target (head):", target_for_model(_ex["target"])[:200], "...")

## 5. Load Qwen2.5-VL-7B in 4-bit (QLoRA base)

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
BASE_TAG = "qwen2.5vl-7b"   # used to name every output dir / Hub repo below

# A single-band crop has FAR fewer pixels than the full page, so the same token budget buys
# much higher effective DPI on the region we care about — the whole point of going regional.
# Bump PROFILE["max_pixels_mult"] (cell 2) if you have VRAM headroom.
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = PROFILE["max_pixels_mult"] * 28 * 28

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=DTYPE,       # float16 on T4, bfloat16 on L4/A100
    bnb_4bit_use_double_quant=True,
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    torch_dtype=DTYPE,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
processor.tokenizer.padding_side = "right"
print("loaded", MODEL_ID, "| max_pixels =", MAX_PIXELS)

## 6. Attach LoRA adapters

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False  # required with gradient checkpointing

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # Adapt the language tower only; leave the vision encoder frozen (cheaper, stabler).
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 7. Collator: image + prompt → masked-label batch

We build Qwen chat messages on the fly (image inline so `process_vision_info` can find
it), render them with the chat template, then mask the prompt-side and image tokens so
loss is computed only on the target JSON.

In [ ]:
from qwen_vl_utils import process_vision_info

IMAGE_PAD_ID = processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")


def to_messages(image, target=None):
    """Qwen chat messages from an already-cropped band image (+ optional target)."""
    msgs = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": PROMPT},
        ],
    }]
    if target is not None:
        msgs.append({"role": "assistant", "content": [{"type": "text", "text": target}]})
    return msgs


def row_to_parts(row, with_target=True):
    """Resolve an index row -> (cropped band image, target-or-None)."""
    ex = ds[row["split"]][int(row["idx"])]
    image = crop_for_model(ex["image"])
    target = target_for_model(ex["target"]) if with_target else None
    return image, target


def collate_fn(rows):
    parts = [row_to_parts(r, with_target=True) for r in rows]
    batch_msgs = [to_messages(img, tg) for img, tg in parts]
    texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in batch_msgs]
    images = []
    for m in batch_msgs:
        imgs, _ = process_vision_info(m)
        images.extend(imgs)
    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)

    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    labels[labels == IMAGE_PAD_ID] = -100  # don't train on vision placeholder tokens
    batch["labels"] = labels
    return batch


# sanity check one batch (also a quick VRAM + sequence-length canary)
_b = collate_fn([train_index[0], train_index[1]])
print({k: tuple(v.shape) for k, v in _b.items() if hasattr(v, "shape")})
print("seq len:", _b["input_ids"].shape[1], "(must be < MAX_LEN below or the target truncates)")

## 8. Train (QLoRA SFT)

7B is slower than v1's 3B — budget **L4 ~2–4 h** for 3 epochs over one band. The adapter
saves every epoch to `qwen2.5vl-7b-hcfa-<band>/`, so a disconnect isn't fatal. CUDA OOM?
Lower `PROFILE["max_pixels_mult"]` in cell 2 and re-run from cell 5.

In [ ]:
from trl import SFTConfig, SFTTrainer

# Truncation ceiling only — dynamic padding keeps real batches as short as their longest
# member. Keep it generous: a full top-band target (~55 keys) or a 6-line service target
# plus vision tokens can exceed 2048, and a truncated label would corrupt training.
MAX_LEN = 4096

args = SFTConfig(
    output_dir=f"{BASE_TAG}-hcfa-{MODEL}",
    num_train_epochs=PROFILE["epochs"],
    per_device_train_batch_size=PROFILE["batch"],
    per_device_eval_batch_size=1,       # eval upcasts logits to fp32; bs>1 OOMs on the loss spike
    gradient_accumulation_steps=PROFILE["grad_accum"],
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    bf16=BF16,                          # auto: bf16 on L4/A100, fp16 on T4
    fp16=not BF16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": True},  # required for Qwen2.5-VL + grad ckpt
    optim="paged_adamw_8bit",           # 8-bit optimizer states — saves VRAM
    logging_steps=5,
    eval_strategy=PROFILE["eval"],
    save_strategy="epoch",
    dataset_kwargs={"skip_prepare_dataset": True},  # our collator does the work
    remove_unused_columns=False,
    report_to="none",
)

# The max-sequence-length field was renamed max_seq_length -> max_length across trl
# versions. Set whichever this install exposes so the notebook is version-proof.
for _name in ("max_length", "max_seq_length"):
    if hasattr(args, _name):
        setattr(args, _name, MAX_LEN)
        break

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_index,
    eval_dataset=val_index if PROFILE["eval"] != "no" else None,
    data_collator=collate_fn,
    processing_class=processor,
)
trainer.train()

## 9. Save / push the adapter

In [ ]:
ADAPTER_DIR  = f"{BASE_TAG}-hcfa-{MODEL}-adapter"
ADAPTER_REPO = f"catochris/{BASE_TAG}-hcfa-{MODEL}-lora"   # private Hub copy = training safety net

trainer.save_model(ADAPTER_DIR)
processor.save_pretrained(ADAPTER_DIR)

# Push the LoRA adapter to the Hub NOW (needs a WRITE token; login() ran in cell 6) so a
# dropped runtime can't cost you the training. The adapter is tiny (~tens of MB).
model.push_to_hub(ADAPTER_REPO, private=True)
processor.push_to_hub(ADAPTER_REPO, private=True)
print("saved adapter ->", ADAPTER_DIR, "| pushed ->", ADAPTER_REPO)

## 10. Inference on the test split

Greedy decode, then pull the JSON object out of the model's text. We keep the raw
output too so the scorer can measure true JSON-validity.

In [ ]:
import json, re

model.config.use_cache = True
model.eval()


def extract_json(text):
    """Parse the model's JSON, salvaging truncated output (keep complete pairs)."""
    t = text.strip()
    if t.startswith("```"):
        t = re.sub(r"^```(?:json)?", "", t).strip().strip("`").strip()
    start = t.find("{")
    if start == -1:
        return None
    frag = t[start:]
    try:
        return json.loads(frag)
    except Exception:
        pass
    # generation got cut off mid-object: keep all complete "key": "value" pairs, close brace
    cut = frag.rfind('",')
    if cut != -1:
        try:
            return json.loads(frag[:cut + 1] + "}")
        except Exception:
            pass
    return None


@torch.no_grad()
def predict_images(images, max_new_tokens=1536):
    """Greedy-generate JSON for several band crops at once.

    1536 covers a 6-line service target; greedy + stop_strings=["}"] halts at the first
    closing brace anyway, so this is just an upper bound.
    """
    msgs = [to_messages(img, None) for img in images]
    texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs]
    vis = []
    for m in msgs:
        imgs, _ = process_vision_info(m)
        vis.extend(imgs)
    processor.tokenizer.padding_side = "left"   # REQUIRED for correct batched generation
    inputs = processor(text=texts, images=vis, return_tensors="pt", padding=True).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        stop_strings=["}"],            # flat JSON ends at one '}' — stop there, not at the cap
        tokenizer=processor.tokenizer,  # required for stop_strings
    )
    gen = out[:, inputs["input_ids"].shape[1]:]
    return [s.strip() for s in processor.batch_decode(gen, skip_special_tokens=True)]


# Sanity check: predict one test form's band crop.
_img, _ = row_to_parts({"idx": 0, "split": "test"}, with_target=False)
_raw = predict_images([_img])[0]
print(f"band '{BAND}' parsed keys:", len(extract_json(_raw) or {}))
print(_raw[:400], "...")

## 11. Score with the repo's `hcfa_eval` harness

We score **this band only**: GT is the flat target subset for `BAND` (`unflatten`-ed for the
harness), and predictions are this model's JSON output per form. Same per-tier /
per-field-class / CER / JSON-validity report as v1, restricted to the fields this model
owns — so you can compare directly against v1's full-page numbers for those same fields.
Because each form yields exactly one JSON object, `parse_rate` is meaningful here.

In [ ]:
from pathlib import Path
from hcfa_eval.schema import unflatten
from hcfa_eval.scoring import summarize, format_summary, write_summary_csv

EVAL_BATCH = 8  # crops per generate() call — lower to 4 if you OOM, raise to 16 if you have headroom

batch_dir = Path(f"eval_batch_{MODEL}"); batch_dir.mkdir(exist_ok=True)
test = ds["test"]
split_rows, pred_lines = [], []

# GT for the harness = only the keys THIS band owns, unflattened.
for ex in test:
    full = json.loads(ex["target"])
    sub = {k: v for k, v in full.items() if regions.region_of_key(k) == BAND}
    (batch_dir / f"{ex['sample_id']}.json").write_text(
        json.dumps({"logical": unflatten(sub)}), encoding="utf-8")
    split_rows.append({"sample_id": ex["sample_id"], "tier": ex["tier"], "json": f"{ex['sample_id']}.json"})

# Predict every test crop (batched).
for i in range(0, len(test), EVAL_BATCH):
    chunk = [test[j] for j in range(i, min(i + EVAL_BATCH, len(test)))]
    images = [crop_for_model(ex["image"]) for ex in chunk]
    for ex, raw in zip(chunk, predict_images(images)):
        parsed = extract_json(raw)
        pred_lines.append(json.dumps({
            "sample_id": ex["sample_id"],
            "fields": parsed if isinstance(parsed, dict) else {},
            "raw": raw,
        }))
    print(f"  predicted {min(i + EVAL_BATCH, len(test))}/{len(test)}")

Path(f"test_split_{MODEL}.jsonl").write_text("\n".join(json.dumps(r) for r in split_rows), encoding="utf-8")
Path(f"preds_{MODEL}.jsonl").write_text("\n".join(pred_lines), encoding="utf-8")

summary = summarize(Path(f"test_split_{MODEL}.jsonl"), Path(f"preds_{MODEL}.jsonl"), batch_dir,
                    model=f"{BASE_TAG}-hcfa-{MODEL}-{PROFILE['tier'].lower()}")
print(format_summary(summary))
write_summary_csv(summary, Path("runs.csv"))
print("\nwrote runs.csv")

## 11b. Merge LoRA → fp16 and push the deployable model

Run this **after** scoring — it frees the 4-bit training model from VRAM, then reloads the
base in fp16 and merges the adapter into one self-contained model (a QLoRA adapter can't be
merged into a 4-bit base). That merged repo is what the Modal `top` worker loads.

> 7B note: we merge on the **GPU** (using the VRAM just freed) because a CPU merge of a 7B
> model needs ~14 GB RAM and can OOM a standard Colab runtime. On an L4/A100 the freed VRAM
> is plenty. If you're on a tiny-VRAM GPU, switch to a **High-RAM** runtime and set
> `device_map="cpu"` below.

In [ ]:
import gc, torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from peft import PeftModel

MERGED_DIR  = f"{BASE_TAG}-hcfa-{MODEL}-merged"
MERGED_REPO = f"catochris/{BASE_TAG}-hcfa-{MODEL}"   # merged = what the Modal '{MODEL}' worker loads

# Free the 4-bit training model + optimizer from VRAM (eval/scoring above is already done).
for _v in ("trainer", "model"):
    if _v in globals():
        del globals()[_v]
gc.collect(); torch.cuda.empty_cache()

# Merge in fp16. device_map="auto" puts the 7B base on the GPU we just freed (a CPU merge of
# 7B needs ~14 GB RAM and can OOM standard Colab). On tiny-VRAM GPUs use a High-RAM runtime
# and set device_map="cpu" instead.
base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=DTYPE, device_map="auto", low_cpu_mem_usage=True,
)
merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
merged.save_pretrained(MERGED_DIR, safe_serialization=True)

# Save the processor WITH the training pixel budget baked in, so serving preprocesses at the
# same resolution the model was trained on (MIN/MAX_PIXELS from cell 11).
proc = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
proc.save_pretrained(MERGED_DIR)

merged.push_to_hub(MERGED_REPO, private=True)
proc.push_to_hub(MERGED_REPO, private=True)
print("merged model ->", MERGED_DIR, "| pushed ->", MERGED_REPO)

## 12. Where to go next

- **Train the other two** — run the `bottom` and `service` notebooks (identical except
  `MODEL`). You'll end up with three merged repos: `catochris/qwen2.5vl-7b-hcfa-top`,
  `-bottom`, `-service`.
- **Serve them concurrently on Modal** — three Modal functions, each loading one merged repo.
  The handler renders the page → crops top/bottom/service (reuse `hcfa_synth.regions`) →
  `.spawn()` all three → joins and merges the JSON. End-to-end latency ≈ the slowest single
  band, not the sum.
- **Compare to v1 fairly** — this report covers only this band; compare those exact fields
  against v1's full-page numbers (watch NPIs / tax IDs / codes — the small-glyph fields the
  crop should most improve).
- **Resolution sweep** — the crop is small, so raise `PROFILE["max_pixels_mult"]` (cell 2)
  as far as VRAM allows; effective DPI on the band is the main accuracy lever.
- **Constrained decoding at serve time** — grammar/JSON-constrained generation (Outlines /
  XGrammar / vLLM structured output) should drive parse failures to ~0; small per-band
  schemas make it cheap.
- **The real gate** — data is 100% synthetic. Validate on real de-identified forms through
  this same harness before trusting production numbers.